# 18.7 教師需要跟著更新嗎？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：老師不只說第一名，也指出接近的候選**

學生選較小架構；老師可能教錯，仍要真值與行爲評估。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/distillation.svg")))

**先想一想：**兩邊都有grad的loss，還是固定教師嗎？

教師是固定指導，不跟着學生loss更新。eval處理模塊行爲，no_grad不建教師圖，requires_grad=False確保teacher權重沒進訓練。

**把直覺寫成數學：**optimizer只收student參數；teacher計算成本仍是真實成本。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

teacher = TinyLM(ModelConfig(width=16)).eval().requires_grad_(False)
student = TinyLM(ModelConfig(width=8))
ids = torch.tensor([[1, 2]])
with torch.no_grad():
    t = teacher(ids)["logits"]
s = student(ids)["logits"]
print(t.requires_grad, s.requires_grad)
assert not t.requires_grad and s.requires_grad

**如何讀結果：**這裏只用隨機模型檢查凍結，不代表這個teacher有可教能力。

**只改一件事：**只取消no_grad但仍freeze，解釋圖何時還會因輸入requires_grad而建立。
